In [ ]:
!pip install -q gradio deep-translator transformers accelerate huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from deep_translator import GoogleTranslator

# 1. Load Gemma (Using 2B instruction-tuned to easily fit in free Colab GPU)
model_id = "google/gemma-1.1-2b-it"
print("Loading model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Uses less memory
    device_map="auto"           # Automatically loads onto the GPU
)

print("Model loaded successfully!")

# 2. Define the core processing function
def process_query(bengali_input):
    try:
        # Step A: Translate Bengali to English
        english_query = GoogleTranslator(source='bn', target='en').translate(bengali_input)

        # Step B: Format for Gemma and Generate
        chat = [{"role": "user", "content": english_query}]
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)

        # Generate the response
        outputs = model.generate(
            inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7
        )

        # Decode only the generated text (ignoring the prompt)
        # Decode only the generated text (ignoring the prompt)
        input_length = inputs.shape[1]
        generated_tokens = outputs[0][input_length:]
        english_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Step C: Translate English Output back to Bengali
        # Note: If the response is too long, deep-translator might truncate it.
        # For a robust app, you'd chunk the text, but this works for standard queries.
        bengali_response = GoogleTranslator(source='en', target='bn').translate(english_response)

        return english_query, english_response, bengali_response

    except Exception as e:
        return f"Error: {str(e)}", f"Error: {str(e)}", f"Error: {str(e)}"

# 3. Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌐 Bengali ↔ Gemma AI Interface")
    gr.Markdown("Type a query in Bengali. The system translates it to English for Gemma, then translates the answer back to Bengali.")

    with gr.Row():
        with gr.Column():
            bn_in = gr.Textbox(label="Bengali Input (আপনার প্রশ্ন)", placeholder="এখানে লিখুন...", lines=4)
            submit_btn = gr.Button("Submit / পাঠান", variant="primary")

        with gr.Column():
            bn_out = gr.Textbox(label="Bengali Output (উত্তর)", interactive=False, lines=5)

    with gr.Accordion("View Translation Pipeline (Under the hood)", open=False):
        en_in = gr.Textbox(label="1. Translated English Query", interactive=False)
        en_out = gr.Textbox(label="2. Gemma's English Response", interactive=False)

    # Trigger the function when button is clicked
    submit_btn.click(
        fn=process_query,
        inputs=bn_in,
        outputs=[en_in, en_out, bn_out]
    )

# Launch the app inline in Colab
demo.launch(debug=True)

Loading model and tokenizer...


config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully!


/tmp/ipykernel_7140/2059785043.py:56: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a9e312a10ffd9b49b8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://899696f1b6cde367c5.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://a9e312a10ffd9b49b8.gradio.live
